# Dense retrieval Example

We will do the following:
- get the text we want to make searchable and apply some light processing to chunk it into sentences.
- embed the sentences.
- build the search index.
- search and see the results.

In [1]:
!pip install cohere
import cohere
import numpy as np
import pandas as pd
from tqdm import tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.0/319.0 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 86.8 MB/s eta 0:00:00


In [2]:
# paste the api key
api_key = "F4fkToUpHPln5emes8HPKUCWzLatSrTXNWerQb5R"

# create and retrieve a cohere api key from os.cohere.ai
co = cohere.Client(api_key)

In [5]:
# getting the text archieve and chunking it .
text = """Interstellar is a 2014 epic science fiction film directed by Christopher Nolan,
who co-wrote the screenplay with his brother Jonathan Nolan.
It features an ensemble cast led by Matthew McConaughey, Anne Hathaway, Jessica Chastain, Bill Irwin, Ellen Burstyn,
and Michael Caine.
Set in a dystopian future where Earth is suffering from catastrophic blight and famine, the film follows a group of
astronauts who travel through a wormhole near Saturn in search of a new home for mankind.
The screenplay had its origins in a script that Jonathan had developed in 2007 and was originally
set to be directed by Steven Spielberg.
Theoretical physicist Kip Thorne was an executive producer and scientific consultant on the film, and wrote the tie-in book
The Science of Interstellar.
It was Lynda Obst's final film as producer before her death.
Cinematographer Hoyte van Hoytema shot it on 35 mm film in the Panavision anamorphic format and
IMAX 70 mm.
Filming began in late 2013 and took place in Alberta, Klaustur, and Los Angeles.
Interstellar uses extensive practical and miniature effects, and the company DNEG created additional visual effects.
Interstellar premiered at the TCL Chinese Theatre on October 26, 2014, and was released in theaters in the
United States on November 5, and in the United Kingdom on November 7, with Paramount Pictures distributing
in the United States and Warner Bros. Pictures distributing in international markets.
In the United States, it was first released on film stock, expanding to venues using digital projectors.
It was a commercial success, grossing $681 million worldwide during its initial theatrical run, and $773.8 million
worldwide with subsequent releases, making it the 10th-highest-grossing film of 2014.
The film received generally positive reviews from critics. Among its various accolades, Interstellar was nominated for five
awards at the 87th Academy Awards, winning Best Visual Effects."""

# split into a list of sentences
texts = text.split('.')
# clean up to remove empty spaces and new lines
texts = [t.strip(' \n') for t in texts]


In [4]:
texts

['Interstellar is a 2014 epic science fiction film directed by Christopher Nolan,\nwho co-wrote the screenplay with his brother Jonathan Nolan',
 'It features an ensemble cast led by Matthew McConaughey, Anne Hathaway, Jessica Chastain, Bill Irwin, Ellen Burstyn,\nand Michael Caine',
 'Set in a dystopian future where Earth is suffering from catastrophic blight and famine, the film follows a group of\nastronauts who travel through a wormhole near Saturn in search of a new home for mankind',
 'The screenplay had its origins in a script that Jonathan had developed in 2007 and was originally\nset to be directed by Steven Spielberg',
 'Theoretical physicist Kip Thorne was an executive producer and scientific consultant on the film, and wrote the tie-in book\nThe Science of Interstellar',
 "It was Lynda Obst's final film as producer before her death",
 'Cinematographer Hoyte van Hoytema shot it on 35 mm film in the Panavision anamorphic format and\nIMAX 70 mm',
 'Filming began in late 2013 a

In [5]:
# embedding the text chunks
# get thet embeddings
response = co.embed(
    texts=texts,
    input_type = "search_document",
).embeddings

In [6]:
embeds = np.array(response)
print(embeds.shape)

(17, 4096)


In [7]:
embeds

array([[ 0.83496094, -0.65966797,  1.8378906 , ...,  1.1728516 ,
        -0.03598022, -1.3417969 ],
       [ 3.9355469 ,  0.5708008 ,  0.6308594 , ...,  0.3395996 ,
         0.27807617, -0.6430664 ],
       [ 1.7626953 , -0.39086914,  1.7197266 , ...,  0.44311523,
        -1.1025391 , -0.0925293 ],
       ...,
       [ 2.2285156 , -0.07617188,  1.5253906 , ..., -1.4316406 ,
         0.24047852,  1.7822266 ],
       [ 1.5195312 , -1.2519531 ,  2.0566406 , ..., -1.4394531 ,
        -1.7001953 , -2.8125    ],
       [ 1.8769531 ,  0.87890625, -0.8178711 , ..., -0.82128906,
         1.3847656 ,  1.3378906 ]])

In [8]:
# building the search index
!pip install faiss-cpu
import faiss
dim = embeds.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(np.float32(embeds))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 41.4 MB/s eta 0:00:00


In [9]:
index

<faiss.swigfaiss_avx2.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFlatL2 *' at 0x7b8dd8a30a20> >

In [10]:
# search the index
# simple embed the query and present its embedding to the index, which will retrieve the most similar sentence from the wikipedia article.
def search(query, number_of_results = 3):
  #1. get the query's embedding
  query_embed = co.embed(texts = [query],
                         input_type = "search_query",).embeddings[0]
  #2. retrieve the nearest neighbors
  distances, similar_items_ids = index.search(np.float32([query_embed]), number_of_results)
  # format the results
  texts_np = np.array(texts) # convert text list to numpy for easier indexing
  results = pd.DataFrame(data={'texts': texts_np[similar_items_ids[0]],
                               'distance': distances[0]})
  # print and return the results
  print(f"Query: '{query}\n Nearest neighbors")
  return results

In [11]:
query = "how precise was the science"
results = search(query)
results

Query: 'how precise was the science
 Nearest neighbors


,texts,distance
0,Interstellar uses extensive practical and mini...,11993.777344
1,Theoretical physicist Kip Thorne was an execut...,12905.022461
2,Cinematographer Hoyte van Hoytema shot it on 3...,13390.091797


In [12]:
# defining a keyword search functino to compare it
# using leading lexical search method - BM25
!pip install rank_bm25
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction import _stop_words
import string

In [13]:
def bm25_tokenizer(text):
  tokenized_doc = []
  for token in text.lower().split():
    token = token.strip(string.punctuation)

    if len(token)>0 and token not in _stop_words.ENGLISH_STOP_WORDS:
      tokenized_doc.append(token)
  return tokenized_doc

In [14]:
tokenized_corpus = []
for passage in tqdm(texts):
  tokenized_corpus.append(bm25_tokenizer(passage))

bm25 = BM25Okapi(tokenized_corpus)

100%|██████████| 17/17 [00:00<00:00, 64998.33it/s]


In [15]:
bm25

In [16]:
def keyword_search(query, top_k=3, num_candidates = 15):
  print(" Input question: ", query)

  #### BM25 search ( lexical search ) #####
  bm25_scores = bm25.get_scores(bm25_tokenizer(query))
  top_n = np.argpartition(bm25_scores, -num_candidates)[-num_candidates:]
  bm25_hits = [{'corpus_id': idx, 'score': bm25_scores[idx]} for idx in top_n]
  bm25_hits = sorted(bm25_hits, key=lambda x: x['score'], reverse=True)
  print(f"BM25 hits for the query: {query}")
  for hit in bm25_hits[0:top_k]:
    print(f"\t {texts[hit['corpus_id']]} (score: {hit['score']:.3f})")

In [17]:
keyword_search("how precise was the science")

 Input question:  how precise was the science
BM25 hits for the query: how precise was the science
	 Theoretical physicist Kip Thorne was an executive producer and scientific consultant on the film, and wrote the tie-in book
The Science of Interstellar (score: 1.647)
	 Interstellar is a 2014 epic science fiction film directed by Christopher Nolan,
who co-wrote the screenplay with his brother Jonathan Nolan (score: 1.647)
	 The screenplay had its origins in a script that Jonathan had developed in 2007 and was originally
set to be directed by Steven Spielberg (score: 0.000)


In [18]:
# limitatino of dense retrieval - n0 answer to the question is found still gives neighbors
query = "what is the mass of the moon?"
results = search(query)
results

Query: 'what is the mass of the moon?
 Nearest neighbors


,texts,distance
0,"8 million\nworldwide with subsequent releases,...",13518.255859
1,"It was a commercial success, grossing $681 mil...",14080.710938
2,Cinematographer Hoyte van Hoytema shot it on 3...,14154.945312


# Reranking Example

In [19]:
query = "how precies was the science"
results = co.rerank(query = query, documents=texts, top_n=3, return_documents=True)
results.results

[RerankResponseResultsItem(document=RerankResponseResultsItemDocument(text='It was a commercial success, grossing $681 million worldwide during its initial theatrical run, and $773'), index=12, relevance_score=0.08306123),
 RerankResponseResultsItem(document=RerankResponseResultsItemDocument(text='8 million\nworldwide with subsequent releases, making it the 10th-highest-grossing film of 2014'), index=13, relevance_score=0.046261825),
 RerankResponseResultsItem(document=RerankResponseResultsItemDocument(text='Theoretical physicist Kip Thorne was an executive producer and scientific consultant on the film, and wrote the tie-in book\nThe Science of Interstellar'), index=4, relevance_score=0.042726286)]

In [20]:
for idx, result in enumerate(results.results):
  print(f"Result {idx}: {result.relevance_score} {result.document.text}")

Result 0: 0.08306123 It was a commercial success, grossing $681 million worldwide during its initial theatrical run, and $773
Result 1: 0.046261825 8 million
worldwide with subsequent releases, making it the 10th-highest-grossing film of 2014
Result 2: 0.042726286 Theoretical physicist Kip Thorne was an executive producer and scientific consultant on the film, and wrote the tie-in book
The Science of Interstellar


In [21]:
# let's see how adding a reranker after a keyword search improves the performance
def keyword_and_reranking_search(query, top_k=3, num_candidates = 10):
  print(" Input question: ", query)
  #### BM25 search ( lexical search ) #####
  bm25_scores = bm25.get_scores(bm25_tokenizer(query))
  top_n = np.argpartition(bm25_scores, -num_candidates)[-num_candidates:]
  bm25_hits = [{'corpus_id': idx, 'score': bm25_scores[idx]} for idx in top_n]
  bm25_hits = sorted(bm25_hits, key=lambda x: x['score'], reverse=True)
  print(f"BM25 hits for the query: {query}")
  for hit in bm25_hits[0:top_k]:
    print(f"\t {texts[hit['corpus_id']]} (score: {hit['score']:.3f})")

  # add reranking
  docs = [texts[hit['corpus_id']] for hit in bm25_hits]
  print(docs)
  print(f"\n Top-3 hits by rank - api ({len(bm25_hits)} bm25 hits re-renked)")
  results = co.rerank(query = query, documents=docs, top_n=top_k, return_documents=True)
  for hit in results.results:
    print(f"\t {hit.document.text.replace("\n", " ")} (score: {hit.relevance_score:.3f})")

In [22]:
keyword_and_reranking_search(query = "how precise was the science")

 Input question:  how precise was the science
BM25 hits for the query: how precise was the science
	 Theoretical physicist Kip Thorne was an executive producer and scientific consultant on the film, and wrote the tie-in book
The Science of Interstellar (score: 1.647)
	 Interstellar is a 2014 epic science fiction film directed by Christopher Nolan,
who co-wrote the screenplay with his brother Jonathan Nolan (score: 1.647)
	 Interstellar uses extensive practical and miniature effects, and the company DNEG created additional visual effects (score: 0.000)
['Theoretical physicist Kip Thorne was an executive producer and scientific consultant on the film, and wrote the tie-in book\nThe Science of Interstellar', 'Interstellar is a 2014 epic science fiction film directed by Christopher Nolan,\nwho co-wrote the screenplay with his brother Jonathan Nolan', 'Interstellar uses extensive practical and miniature effects, and the company DNEG created additional visual effects', 'It was a commercial s

# RAG

## Example : grounded Generation with an llp api


In [23]:
query = "income generated"

#1- retrieval
restults = search(query)

# 2 grounded generation
docs_dict = [{'text': text} for text in restults['texts']]
response = co.chat(
    message=query,
    documents= docs_dict
)

Query: 'income generated
 Nearest neighbors


In [24]:
print(response.text)

The film Interstellar generated $681 million worldwide during its initial theatrical run, and $773 million worldwide with subsequent releases.


## Example: RAG with Local Models

In [1]:
# loading the generation model
!wget https://huggingface.co/microsoft/Phi-3-mini-4k-instruct-gguf/resolve/main/Phi-3-mini-4k-instruct-fp16.gguf

--2026-01-11 05:04:17--  https://huggingface.co/microsoft/Phi-3-mini-4k-instruct-gguf/resolve/main/Phi-3-mini-4k-instruct-fp16.gguf
Resolving huggingface.co (huggingface.co)... 3.165.160.61, 3.165.160.59, 3.165.160.12, ...
Connecting to huggingface.co (huggingface.co)|3.165.160.61|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/662698108f7573e6a6478546/a9cdcf6e9514941ea9e596583b3d3c44dd99359fb7dd57f322bb84a0adc12ad4?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27Phi-3-mini-4k-instruct-fp16.gguf%3B+filename%3D%22Phi-3-mini-4k-instruct-fp16.gguf%22%3B&Expires=1768111457&Policy=eyJTdGF0ZW1lbnQiOlt7IkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiRXBvY2hUaW1lIjoxNzY4MTExNDU3fX0sIlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjYyNjk4MTA4Zjc1NzNlNmE2NDc4NTQ2L2E5Y2RjZjZlOTUxNDk0MWVhOWU1OTY1ODNiM2QzYzQ0ZGQ5OTM1OWZiN2RkNTdmMzIyYmI4NGEwYWRjMTJhZDRcXD9yZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPSoifV19&Sign

In [2]:
!pip install langchain-community langchain
!pip install llama-cpp-python
from langchain_community.llms import LlamaCpp

# make sure the model path is correct for your system!
llm = LlamaCpp(
    model_path = "/content/Phi-3-mini-4k-instruct-fp16.gguf",
    n_gpu_layers=-1,
    max_tokens = 500,
    n_ctx=2048,
    seed = 42,
    verbose= False
)

llama_context: n_batch is less than GGML_KQ_MASK_PAD - increasing to 64
llama_context: n_ctx_per_seq (2048) < n_ctx_train (4096) -- the full capacity of the model will not be utilized


In [3]:
# loading the embedding model
from langchain_community.embeddings.huggingface import HuggingFaceEmbeddings

# embedding model for converting text to numerical representations
embedding_model = HuggingFaceEmbeddings(model_name="thenlper/gte-small")

/tmp/ipython-input-2946778467.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="thenlper/gte-small")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [6]:
# now use the embedding model to set up our vector database
from langchain_community.vectorstores import FAISS

# create a local vector database
db = FAISS.from_texts(texts, embedding_model)

In [15]:
# the rag prompt
from langchain_core.prompts import ChatPromptTemplate

In [26]:
# create a prompt template
template = """<|user|>
Relevant information:
{context}

Provide a concise answer to the following querstion using the relevant information provided above:
{question}<|end|>
<|assistant|>
"""
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", template),
        ("human", "{input}"),
    ]
)

In [27]:
#!pip install --upgrade --force-reinstall langchain
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
# rag pipeline
retriever = db.as_retriever()
question_answer_chain = create_stuff_documents_chain(llm, prompt=prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

# 4. Invoke the chain
response = rag_chain.invoke({"input": "Income generated"})
print(response["answer"])

KeyError: "Input to ChatPromptTemplate is missing variables {'question'}.  Expected: ['context', 'input', 'question'] Received: ['input', 'context']\nNote: if you intended {question} to be part of the string and not a variable, please escape it with double curly braces like: '{{question}}'.\nFor troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/INVALID_PROMPT_INPUT "